In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import plotly.express as px
from mpl_toolkits.mplot3d import Axes3D

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
import cv2

from sklearn.manifold import TSNE
import umap

In [ ]:
df = pd.read_csv('dataset_cleaned.csv')

## Distribusi Kelas

In [ ]:
class_counts = df['label'].value_counts().reset_index()
class_counts.columns = ['Kelas', 'Jumlah']
class_counts['Persentase'] = (class_counts['Jumlah'] / class_counts['Jumlah'].sum() * 100).round(1)

print("Distribusi Kelas (setelah cleaning):")
print(class_counts.to_string(index=False))

In [ ]:
sns.set_style("whitegrid")

palette_colors = sns.color_palette("Set1", n_colors=len(class_counts))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = sns.barplot(data=class_counts, x='Kelas', y='Jumlah',hue='Kelas', palette=palette_colors, ax=axes[0])
ax1.set_title('Jumlah Gambar per Kelas', fontsize=13, fontweight='bold')
for i, row in class_counts.iterrows():
    ax1.text(i, row['Jumlah'] + 5, f"{row['Jumlah']} ({row['Persentase']}%)",
             ha='center', fontsize=10)

axes[1].pie(class_counts['Jumlah'], labels=class_counts['Kelas'], autopct='%1.1f%%',
            startangle=90, explode=(0.02, 0.02, 0.02), colors=palette_colors)
axes[1].set_title('Proporsi Kelas', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig("class_distribution_seaborn.png", dpi=150)
plt.show()


## Analisis Rasio

In [ ]:
def dimensi(path):
    """
    Mengambil ukuran dari gambar

    :param path: path gambar
    :return: ukurang gambar dalam bentuk (panjang, lebar)
    """
    try:
        with Image.open(path) as img:
            return img.size  # (width, height)
    except Exception:
        return None

paths = df['file_path'].tolist()
dims = [None] * len(paths)


In [ ]:
with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_idx = {executor.submit(dimensi, p): i for i, p in enumerate(paths)}
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            dims[idx] = future.result()
        except:
            pass

In [ ]:
df['width'] = [d[0] if d else None for d in dims]
df['height'] = [d[1] if d else None for d in dims]
df.dropna(subset=['width', 'height'], inplace=True)

df['aspect_ratio'] = df['width'] / df['height']

print("\nStatistik Dimensi Gambar:")
print(df[['width', 'height', 'aspect_ratio']].describe())

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(28, 20))

sns.histplot(df['width'], bins=50, kde=True, color='steelblue', ax=axes[0,0])
axes[0,0].set_title('Distribusi Lebar Gambar', fontweight='bold')
axes[0,0].set_xlabel('Width (piksel)')

sns.histplot(df['height'], bins=50, kde=True, color='coral', ax=axes[0,1])
axes[0,1].set_title('Distribusi Tinggi Gambar', fontweight='bold')
axes[0,1].set_xlabel('Height (piksel)')

if 'label' in df.columns:
    scatter = sns.scatterplot(data=df, x='width', y='height', hue='label', alpha=0.5, s=20, ax=axes[1,0])
    scatter.legend(title='Kelas', loc='upper left', bbox_to_anchor=(1,1))
else:
    sns.scatterplot(data=df, x='width', y='height', alpha=0.5, s=20, ax=axes[1,0])
axes[1,0].set_title('Width vs Height', fontweight='bold')
axes[1,0].plot([0, df['width'].max()], [0, df['width'].max()], 'k--', alpha=0.3, label='Square (1:1)')
axes[1,0].set_xlabel('Width')
axes[1,0].set_ylabel('Height')

sns.histplot(df['aspect_ratio'], bins=50, kde=True, color='mediumseagreen', ax=axes[1,1])
axes[1,1].set_title('Distribusi Rasio Aspek (Width/Height)', fontweight='bold')
axes[1,1].axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='1:1 (persegi)')
axes[1,1].legend()

plt.tight_layout()
plt.savefig("resolution_aspect_analysis.png", dpi=150)
plt.show()

outliers = df[(df['width'] < 50) | (df['height'] < 50) | (df['width'] > 5000) | (df['height'] > 5000)]
if len(outliers) > 0:
    print(f"\nDitemukan {len(outliers)} gambar dengan dimensi ekstrim:")
    print(outliers[['file_path', 'width', 'height']].head(10).to_string(index=False))
else:
    print("\nTidak ada gambar dengan dimensi ekstrim (<50px atau >5000px).")

- Sebagian besar gambar memiliki dimensi kecil (median width 254px, height 194px) dengan variasi yang sangat besar (hingga 8000×6000px).

- Distribusi lebar, tinggi, dan rasio aspek bersifat right‑skewed, didominasi oleh gambar beresolusi rendah hingga sedang.

- Rasio aspek mayoritas mendekati persegi (median 1.26), namun terdapat gambar dengan orientasi landscape atau portrait ekstrem (rasio hingga 8.7).

- Kelas Electronic menunjukkan variasi dimensi paling luas dan mencakup seluruh outlier dimensi tinggi, sementara kelas Organic dan Recyclable cenderung beresolusi lebih kecil dan seragam.

- Teridentifikasi 16 gambar berdimensi >5000px pada kelas Electronic

## Contoh per Kelas

In [ ]:
n_sample = 5
classes = df['label'].unique()

In [ ]:
fig, axes = plt.subplots(len(classes), n_sample, figsize=(n_sample * 2.5, len(classes) * 2.5))

# fig.suptitle("Contoh Gambar per Kelas", fontsize=20, fontweight='bold')

for i, cls in enumerate(classes):
    file_kelas = df[df['label'] == cls]['file_path'].tolist()
    samples = random.sample(file_kelas, n_sample)

    for j, filepath in enumerate(samples):
        ax = axes[i,j]
        img = Image.open(filepath)
        ax.imshow(img)
        ax.set_title(f"{cls}\n{Path(filepath).name[:20]}...", fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=1)
plt.show()

## Intensitas Warna per Kelas

In [ ]:
def mean_rgb(path):
    """
    Menghitung mean masing-masing channel pada RGB

    :param path:  path gambar
    :return: rata-rata RGB (r,g,b)
    """
    img = Image.open(path).convert('RGB')
    arr = np.array(img)
    r,g,b = arr[:,:,0].mean(), arr[:,:,1].mean(), arr[:,:,2].mean()
    return (r,g,b)

In [ ]:
sns.set_style("whitegrid")
df_sample = df.copy()


paths = df_sample['file_path'].tolist()
results = [None] * len(paths)

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_idx = {executor.submit(mean_rgb, p): i for i, p in enumerate(paths)}
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        try:
            results[idx] = future.result()
        except:
            pass

df_sample['R'] = [r[0] if r else None for r in results]
df_sample['G'] = [r[1] if r else None for r in results]
df_sample['B'] = [r[2] if r else None for r in results]
df_sample['Brightness'] = (df_sample['R'] + df_sample['G'] + df_sample['B']) / 3
df_sample.dropna(subset=['R', 'G', 'B'], inplace=True)

classes = sorted(df_sample['label'].unique())
fig = plt.figure(figsize=(16, 12))

# Subplot 1: Histogram brightness per kelas w KDE
ax1 = fig.add_subplot(2, 2, 1)
for cls in classes:
    subset = df_sample[df_sample['label'] == cls]
    sns.histplot(subset['Brightness'], kde=True, label=cls,
                 fill=True, alpha=0.2, ax=ax1, element="step")
ax1.legend(title='Kelas')
ax1.set_title('Distribusi Brightness per Kelas (KDE)', fontweight='bold')
ax1.set_xlabel('Brightness (0-255)')

# Subplot 2: Boxplot brightness per kelas
ax2 = fig.add_subplot(2, 2, 2)
sns.boxplot(data=df_sample, x='label', y='Brightness',hue='label', palette='Set2', ax=ax2)
ax2.set_title('Boxplot Brightness per Kelas', fontweight='bold')

# Subplot 3: Rata-rata RGB per kelas (bar chart)
ax3 = fig.add_subplot(2, 2, 3)
mean_rgb_df = df_sample.groupby('label')[['R', 'G', 'B']].mean()
x = np.arange(len(mean_rgb_df))
width = 0.25
for j, (ch, color) in enumerate(zip(['R', 'G', 'B'], ['red', 'green', 'blue'])):
    ax3.bar(x + j*width, mean_rgb_df[ch], width, color=color, alpha=0.7, label=ch)
ax3.set_xticks(x + width)
ax3.set_xticklabels(mean_rgb_df.index)
ax3.set_title('Rata-rata Intensitas RGB per Kelas', fontweight='bold')
ax3.set_ylabel('Intensitas')
ax3.legend()

# Subplot 4: 3D Scatter R, G, B
ax4 = fig.add_subplot(2, 2, 4, projection='3d')
colors = sns.color_palette("Set2", n_colors=len(classes))
for cls, color in zip(classes, colors):
    subset = df_sample[df_sample['label'] == cls]
    ax4.scatter(subset['R'], subset['G'], subset['B'],
                c=[color], label=cls, alpha=0.3, s=10, marker='o')
ax4.set_xlabel('R')
ax4.set_ylabel('G')
ax4.set_zlabel('B')
ax4.set_title('Scatter 3D Rata-rata RGB per Gambar')
ax4.legend()

plt.tight_layout()
plt.savefig("color_analysis_with_3d.png", dpi=150)
plt.show()

In [ ]:
# Statistik numerik
print("\nRata-rata intensitas warna per kelas:")
print(mean_rgb_df.round(1))
print("\nRata-rata brightness per kelas:")
print(df_sample.groupby('label')['Brightness'].mean().round(1))

## Analisis t-SNE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan perangkat: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['file_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, row['label']

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # default di resnet
                         std=[0.229, 0.224, 0.225])
])

dataset = ImageDataset(df, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

In [ ]:
# Model ResNet50
weights = models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)
model = torch.nn.Sequential(*list(model.children())[:-1])
model = model.to(device)
model.eval()

# Ekstraksi Fitur
features = []
labels = []
with torch.no_grad():
    for imgs, lbls in tqdm(dataloader, desc="Ekstraksi fitur"):
        imgs = imgs.to(device)
        feats = model(imgs).squeeze(-1).squeeze(-1)   # [batch, 2048]
        features.append(feats.cpu().numpy())
        labels.extend(lbls)

features = np.concatenate(features, axis=0)
labels = np.array(labels)
print(f"Fitur berhasil diekstrak: {features.shape}")

# Reduksi Dimensi w UMAP
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, verbose=1)
# Parameter penting: n_neighbors (keseimbangan lokal/global), min_dist (kerapatan klaster)

features_2d = reducer.fit_transform(features)

df_umap = pd.DataFrame({
    'x': features_2d[:, 0],
    'y': features_2d[:, 1],
    'label': labels
})


In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(12, 10))
classes = sorted(df_umap['label'].unique())
palette = sns.color_palette("Set2", n_colors=len(classes))

for cls, color in zip(classes, palette):
    subset = df_umap[df_umap['label'] == cls]
    plt.scatter(subset['x'], subset['y'], c=[color], label=cls, alpha=0.4, s=5)

plt.title('Proyeksi UMAP Fitur Global (ResNet50)', fontsize=14, fontweight='bold')
plt.xlabel('Komponen UMAP 1')
plt.ylabel('Komponen UMAP 2')
plt.legend(title='Kelas', markerscale=4)
plt.tight_layout()
plt.savefig("umap_global_features.png", dpi=150)
plt.show()

## Saliency & Gradcam pada Model Probe (Resnet18)

In [ ]:
label_map = {name: idx for idx, name in enumerate(sorted(df['label'].unique()))}
idx_to_label = {v: k for k, v in label_map.items()}
print("Label mapping:", label_map)

In [ ]:
class ImageDataset(Dataset):
    """Sama seperti di umap beda hanya label diubah dengan torch.tensor"""
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['file_path']).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), (0,0,0))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label_map[row['label']])

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # default di resnet
                         std=[0.229, 0.224, 0.225])
])


dataset = ImageDataset(df, transform=transform)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# Model Baseline (ResNet-18)
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(label_map))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/5"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"  Loss: {running_loss/len(train_loader):.4f}")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
print(f"Akurasi validasi: {100*correct/total:.2f}%")

In [ ]:
def grad_cam_cv2(model, img_path, target_layer='layer4'):
    """
    Grad-CAM untuk ResNet18, layer target = layer4.

    :param model:
    """
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (224, 224))
    img_tensor = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)

    model.eval()
    features, gradients = [], []
    def forward_hook(module, input, output):
        features.append(output)
    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])
    target = dict([*model.named_modules()])[target_layer]
    fh = target.register_forward_hook(forward_hook)
    bh = target.register_backward_hook(backward_hook)

    output = model(img_tensor)
    pred_class = output.argmax(dim=1).item()
    model.zero_grad()
    one_hot = torch.zeros_like(output)
    one_hot[0, pred_class] = 1
    output.backward(gradient=one_hot)

    fh.remove(); bh.remove()
    pooled_grad = torch.mean(gradients[0], dim=[2,3], keepdim=True)
    cam = torch.sum(pooled_grad * features[0], dim=1).squeeze()
    cam = torch.relu(cam).cpu().detach().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (224, 224))
    heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlayed = cv2.addWeighted(img_resized, 0.6, heatmap, 0.4, 0)
    return overlayed, idx_to_label[pred_class]

In [ ]:
def saliency_map(model, img_path):
    """
    Vanilla saliency map, mengembalikan array 2D (H,W) 0-1.
    """
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    model.eval()
    output = model(img_tensor)
    pred_class = output.argmax(dim=1).item()
    model.zero_grad()
    one_hot = torch.zeros_like(output)
    one_hot[0, pred_class] = 1
    output.backward(gradient=one_hot)
    # Ambil nilai maksimum gradien di antara channel RGB -> saliency 2D
    saliency = img_tensor.grad.abs().max(dim=1)[0]  # [1, H, W]
    saliency = saliency.squeeze().cpu().numpy()
    # Normalisasi
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency, idx_to_label[pred_class]

In [ ]:
samples_list = []
for cls in df['label'].unique():
    cls_df = df[df['label'] == cls].sample(5)
    samples_list.append(cls_df)
samples_df = pd.concat(samples_list).reset_index(drop=True)

fig, axes = plt.subplots(len(samples_df), 3, figsize=(12, 4*len(samples_df)))
for i, row in tqdm(samples_df.iterrows(), total=len(samples_df)):
    img_path = row['file_path']
    true_label = row['label']

    cam_overlay, cam_pred = grad_cam_cv2(model, img_path)
    sal, sal_pred = saliency_map(model, img_path)
    original = cv2.imread(str(img_path))
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    original = cv2.resize(original, (224,224))

    axes[i,0].imshow(original)
    axes[i,0].set_title(f"Asli: {true_label}", fontsize=8)
    axes[i,0].axis('off')
    axes[i,1].imshow(cam_overlay)
    axes[i,1].set_title(f"Grad-CAM pred: {cam_pred}", fontsize=8)
    axes[i,1].axis('off')
    axes[i,2].imshow(sal, cmap='hot')
    axes[i,2].set_title(f"Saliency pred: {sal_pred}", fontsize=8)
    axes[i,2].axis('off')

plt.tight_layout()
plt.savefig("gradcam_saliency_eda.png", dpi=150)
plt.show()